# FiratSuper Flux generate — **step2**

This notebook is a **generate-only continuation**. Training and archive history stay in `Flux_LoRA_Training_Colab.ipynb`. **Do not retrain here.** Same Drive / LoRA paths under `MyDrive/FiratSuper/...`.

**Runtime:** Runtime > Change runtime type > **A100 GPU**. Do not pick T4. Do not pick TPU.

**Drive:** Chrome, one Google account only (`superweb.contact@gmail.com`). In the popup: Continue, then **Allow ALL** permissions.

**Hugging Face:** Accept the license for [black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev) and paste a READ token.

**Locked LoRA (do not overwrite):** `MyDrive/FiratSuper/loras/lapetitemilf_flux_v2.safetensors`
Also locked: `lapetitemilf_flux` (v1) and `lapetitemilf_face`. Never write `.safetensors` from this notebook.

Cells **1–4** match the training notebook (GPU check, Drive mount, HF login, install) so setup muscle memory stays the same. Skip every training cell in the old notebook. Then run **one** experiment cell, or the chest-LoRA pair: **8** (train chest adapter from real `chest_real` only) then **9** (generate v2 + chest LoRA). Cells **5–7** stay as rejected/history (faces OK; nipples not locked).

**Trigger:** `ohwx woman` (face LoRA, locked). Chest adapter trigger: `chtreal`. Adult subject only. Do not train on generated pictures.

## Cells
1. A100 GPU check
2. Drive + settings
3. Hugging Face login
4. Install (needed on a fresh runtime; skip if packages are already in this session)
5. Woman-only face lock + photoreal (no couple, no paste)
6. Woman-only bare chest QA (face lock from 5, no bra, no paste) — nipples REJECTED
7. Weak img2img from approved keepers 17+18 (preserve chest) — nipples REJECTED
8. Train small chest/nipple LoRA from chest_real only (does not touch v2)
9. Woman-only generate with v2 + chest LoRA (no paste, no male)

## Drive layout
```
MyDrive/FiratSuper/
|-- loras/lapetitemilf_flux_v2.safetensors       # LOCKED face — load only
|-- loras/lapetitemilf_chest_flux_v1.safetensors # cell 8 chest adapter
|-- ADD_FLUX_CHEST/                             # real chest inbox (optional extra)
|-- generate/                                   # step2 stills — NEVER train on these
`-- keepers/chest_real/                         # canonical REAL chest photos only
```


In [ ]:
# @title 1) A100 GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Runtime > Change runtime type > A100 GPU. Do not pick TPU, then rerun."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu)
print("VRAM: %.1f GB" % vram)

name = gpu.upper()
if "TPU" in name:
    raise RuntimeError("TPU is not supported. Runtime > Change runtime type > A100 GPU.")
if ("A100" not in name) and ("H100" not in name):
    raise RuntimeError(
        "This notebook needs Colab Pro A100 (or H100). Got: %s (%.1f GB). "
        "Runtime > Change runtime type > A100 GPU. T4 and L4 are too small."
        % (gpu, vram)
    )
if vram < 35:
    raise RuntimeError(
        "A100-class GPU but VRAM is %.1f GB. Need ~40 GB. Change runtime type."
        % vram
    )
print("A100 check OK.")

In [ ]:
# @title 2) Connect Drive + project settings
from google.colab import auth, drive
import os
import shutil

MOUNT = "/content/drive"
MYDRIVE = os.path.join(MOUNT, "MyDrive")
FIRATSUPER_DRIVE_ID = "18UE4fijDjq8ggmkDYjaUpRQXVDE0cqYt"
FLUX_INBOX_ID = "1oLtTmwg2kt-Jn6zuci06ipRQoK6AOFVZ"
FLUX_CHEST_ID = "1iEmUvagFQVJ2TArN_7ee4Af4TUti1hZw"
HENRY_BODY_ID = "1CmFmJVtOW-a39rRndSZ4PDJc8iJIX3sm"
USE_DRIVE_API = False
DRIVE_SERVICE = None


def _drive_ok():
    return os.path.isdir(MYDRIVE)


def _mount_fuse(force=False):
    if _drive_ok() and not force:
        return True
    print("Drive popup: click Continue, then Allow ALL permissions. Do not uncheck boxes.")
    try:
        drive.mount(MOUNT, force_remount=force)
    except Exception as err:
        print("drive.mount failed:", err)
    return _drive_ok()


def _google_login():
    print("Google login popup (not the Drive FUSE popup)...")
    try:
        auth.authenticate_user()
        print("Google login OK")
        return True
    except Exception as err:
        print("Google login failed:", err)
        return False


def _api_service():
    from googleapiclient.discovery import build
    return build("drive", "v3")


def api_find_child(service, parent_id, name):
    q = "'" + parent_id + "' in parents and name = '" + name + "' and trashed = false"
    resp = service.files().list(
        q=q,
        fields="files(id, name, mimeType)",
        pageSize=10,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()
    files = resp.get("files", [])
    return files[0] if files else None


def api_ensure_folder(service, parent_id, name):
    found = api_find_child(service, parent_id, name)
    if found:
        return found["id"]
    meta = {
        "name": name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id],
    }
    return service.files().create(
        body=meta, fields="id", supportsAllDrives=True
    ).execute()["id"]


def api_list_children(service, folder_id):
    items = []
    token = None
    while True:
        resp = service.files().list(
            q="'" + folder_id + "' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=token,
            pageSize=1000,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        items.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            break
    return items


def api_download_file(service, file_id, dest):
    from googleapiclient.http import MediaIoBaseDownload
    parent = os.path.dirname(dest)
    if parent:
        os.makedirs(parent, exist_ok=True)
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _status, done = downloader.next_chunk()


def api_download_folder(service, folder_id, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    for item in api_list_children(service, folder_id):
        path = os.path.join(dest_dir, item["name"])
        if item["mimeType"] == "application/vnd.google-apps.folder":
            api_download_folder(service, item["id"], path)
        else:
            print("  copy", item["name"])
            api_download_file(service, item["id"], path)


def api_upload_file(service, local_path, parent_id, name):
    from googleapiclient.http import MediaFileUpload
    if name in PROTECTED_LORAS:
        raise RuntimeError("Refusing to overwrite protected LoRA: " + name)
    media = MediaFileUpload(local_path, resumable=True)
    found = api_find_child(service, parent_id, name)
    if found:
        if found["name"] in PROTECTED_LORAS:
            raise RuntimeError("Refusing to overwrite protected LoRA: " + name)
        service.files().update(
            fileId=found["id"], media_body=media, supportsAllDrives=True
        ).execute()
        return found["id"]
    body = {"name": name, "parents": [parent_id]}
    created = service.files().create(
        body=body, media_body=media, fields="id", supportsAllDrives=True
    ).execute()
    return created["id"]


def upload_project_file(local_path, dest_rel=None):
    if dest_rel is None:
        dest_rel = os.path.relpath(local_path, ROOT)
    if USE_DRIVE_API:
        if DRIVE_SERVICE is None:
            print("Skip Drive upload: no API client")
            return
        print("Uploading to Drive:", dest_rel)
        parts = dest_rel.split("/")
        parent = FIRATSUPER_DRIVE_ID
        for folder in parts[:-1]:
            parent = api_ensure_folder(DRIVE_SERVICE, parent, folder)
        api_upload_file(DRIVE_SERVICE, local_path, parent, parts[-1])
        print("Uploaded:", dest_rel)
        return
    dest = os.path.join(ROOT, dest_rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.abspath(local_path) != os.path.abspath(dest):
        shutil.copy2(local_path, dest)
    print("Saved:", dest)


def sync_flux_via_api(local_root):
    print("FUSE mount failed. Copying Flux folders via Drive API to", local_root)
    service = _api_service()
    os.makedirs(local_root, exist_ok=True)
    inbox = api_find_child(service, FIRATSUPER_DRIVE_ID, "ADD_FLUX_PHOTOS")
    inbox_id = inbox["id"] if inbox else FLUX_INBOX_ID
    api_download_folder(
        service,
        inbox_id,
        os.path.join(local_root, "ADD_FLUX_PHOTOS"),
    )
    chest = api_find_child(service, FIRATSUPER_DRIVE_ID, "ADD_FLUX_CHEST")
    chest_id = chest["id"] if chest else FLUX_CHEST_ID
    api_download_folder(
        service,
        chest_id,
        os.path.join(local_root, "ADD_FLUX_CHEST"),
    )
    henry = api_find_child(service, FIRATSUPER_DRIVE_ID, "ADD_HENRY_BODY_PHOTOS")
    henry_id = henry["id"] if henry else HENRY_BODY_ID
    api_download_folder(
        service,
        henry_id,
        os.path.join(local_root, "ADD_HENRY_BODY_PHOTOS"),
    )
    for sub in ("output", "loras", "logs", "keepers"):
        os.makedirs(os.path.join(local_root, sub), exist_ok=True)
    return service


if _drive_ok():
    print("Drive already mounted at", MYDRIVE)
elif _mount_fuse(force=False):
    print("Drive mounted at", MYDRIVE)
else:
    print("FUSE mount failed. Trying Google login, then mount again...")
    logged_in = _google_login()
    if logged_in and _mount_fuse(force=True):
        print("Drive mounted after Google login:", MYDRIVE)
    elif logged_in:
        USE_DRIVE_API = True
        DRIVE_SERVICE = sync_flux_via_api("/content/FiratSuper")
        print("DRIVE MODE: API fallback (local /content/FiratSuper)")
        print("LoRA and previews will upload back to Drive after training.")
    else:
        print("Could not connect to Drive.")
        print("1. Left sidebar: folder icon -> Mount Drive, then rerun this cell")
        print("2. Chrome, one Google account only (the account that owns the photos)")
        print("3. Allow ALL permissions. Do not close extra popups")
        print("4. Runtime > Disconnect and delete runtime, reconnect A100 GPU")
        raise RuntimeError("Drive is not connected. See the steps printed above.")

if (not USE_DRIVE_API) and (not _drive_ok()):
    raise RuntimeError("Drive folder is empty. Grant access and rerun this cell.")

# === edit here ===
PROJECT_NAME = "lapetitemilf"
TRIGGER_WORD = "ohwx woman"
LORA_NAME = "lapetitemilf_flux_v2"
EXPECTED_PAIRS = 38
TRAIN_STEPS = 2000
DRY_RUN_STEPS = 5
NETWORK_DIM = 16
NETWORK_ALPHA = 16
LEARNING_RATE = 1e-4
SAVE_EVERY = 500
SAMPLE_EVERY = 500
SUBJECT_IS_ADULT = True
# =================

PROTECTED_LORAS = {
    "lapetitemilf_face.safetensors",
    "lapetitemilf_body.safetensors",
    "lapetitemilf_thorough.safetensors",
    "lapetitemilf_standard.safetensors",
    "lapetitemilf_lora.safetensors",
    "lapetitemilf_together.safetensors",
    "lapetitemilf_flux.safetensors",
    "lapetitemilf_flux_v2.safetensors",
}
OUTPUT_LORA_NAME = LORA_NAME + ".safetensors"
if OUTPUT_LORA_NAME in PROTECTED_LORAS:
    print("LoRA is LOCKED:", OUTPUT_LORA_NAME)
    print("Do not retrain. Generate with step2 cell 5.")
if not SUBJECT_IS_ADULT:
    raise RuntimeError("This notebook is for an adult subject only.")

if USE_DRIVE_API:
    ROOT = "/content/FiratSuper"
else:
    ROOT = "/content/drive/MyDrive/FiratSuper"

INBOX_DIR = os.path.join(ROOT, "ADD_FLUX_PHOTOS")
CHEST_DIR = os.path.join(ROOT, "ADD_FLUX_CHEST")
HENRY_INBOX_DIR = os.path.join(ROOT, "ADD_HENRY_BODY_PHOTOS")
LORAS_DIR = os.path.join(ROOT, "loras")
KEEPERS_DIR = os.path.join(ROOT, "keepers")
EVAL_DIR = os.path.join(ROOT, "output", PROJECT_NAME, "flux_eval_v2")
SAMPLES_DIR = os.path.join(ROOT, "output", PROJECT_NAME, "flux_samples_v2")
DATASET_DIR = "/content/dataset"
TRAIN_OUTPUT_DIR = "/content/output"
CONFIG_PATH = "/content/lapetitemilf_flux.yaml"
HENRY_LORA_NAME = "henry_penis_flux_v1"
HENRY_TRIGGER = "hrmale"
HENRY_EXPECTED = 26
HENRY_TRAIN_STEPS = 2000
HENRY_DATASET_DIR = "/content/dataset_henry"
HENRY_CONFIG_PATH = "/content/henry_penis_flux.yaml"
HENRY_OUTPUT_LORA = HENRY_LORA_NAME + ".safetensors"
if HENRY_OUTPUT_LORA in PROTECTED_LORAS:
    raise RuntimeError("Male LoRA name is on the lock list. Pick a new filename.")
if HENRY_LORA_NAME == LORA_NAME or HENRY_OUTPUT_LORA == OUTPUT_LORA_NAME:
    raise RuntimeError("Male LoRA must not reuse the locked v2 filename.")
os.makedirs(LORAS_DIR, exist_ok=True)
os.makedirs(KEEPERS_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(HENRY_DATASET_DIR, exist_ok=True)
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)

free_gb = shutil.disk_usage("/content").free / 1024**3
print("ROOT:", ROOT)
print("Inbox:", INBOX_DIR)
print("Chest:", CHEST_DIR)
print("Henry inbox:", HENRY_INBOX_DIR)
print("Local dataset:", DATASET_DIR)
print("Henry dataset:", HENRY_DATASET_DIR)
print("LoRA out:", os.path.join(LORAS_DIR, OUTPUT_LORA_NAME), "(LOCKED)")
print("Male LoRA out:", os.path.join(LORAS_DIR, HENRY_OUTPUT_LORA))
print("Free disk: %.1f GB" % free_gb)
if free_gb < 40:
    raise RuntimeError("Need ~40 GB free for Flux.1-dev. Have %.1f GB." % free_gb)


def _purge_old_torchao():
    import sys
    import subprocess
    import importlib
    try:
        from importlib.metadata import version
        ver = version("torchao")
    except Exception:
        print("torchao not installed")
        return
    print("torchao", ver)
    nums = []
    for part in ver.split("."):
        digits = "".join(ch for ch in part if ch.isdigit())
        if digits:
            nums.append(int(digits))
    while len(nums) < 3:
        nums.append(0)
    if tuple(nums[:3]) >= (0, 16, 0):
        return
    print("peft needs torchao>=0.16 to load Flux LoRA. Uninstalling", ver)
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
    for key in list(sys.modules):
        if key == "torchao" or key.startswith("torchao.") or key.startswith("peft.tuners.lora.torchao"):
            del sys.modules[key]
    if "peft.import_utils" in sys.modules:
        importlib.reload(sys.modules["peft.import_utils"])
    print("old torchao removed")


def ensure_flux_pipe():
    global pipe
    import gc
    import os
    import torch
    need_load = True
    if "pipe" in globals() and pipe is not None:
        if type(pipe).__name__ == "FluxPipeline":
            need_load = False
            print("Using Flux txt2img already in memory.")
        else:
            print("In-memory pipe is", type(pipe).__name__, "- loading txt2img.")
    if not need_load:
        return pipe
    lora_path = os.path.join(LORAS_DIR, OUTPUT_LORA_NAME)
    if not os.path.isfile(lora_path):
        local_final = os.path.join(TRAIN_OUTPUT_DIR, LORA_NAME, OUTPUT_LORA_NAME)
        if os.path.isfile(local_final):
            lora_path = local_final
    if not os.path.isfile(lora_path):
        raise RuntimeError("LoRA not found: " + lora_path)
    gc.collect()
    torch.cuda.empty_cache()
    _purge_old_torchao()
    from diffusers import FluxPipeline
    print("Loading FLUX.1-dev + LoRA (no safety checker)...")
    print("LoRA file:", lora_path)
    loaded = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev",
        torch_dtype=torch.bfloat16,
        token=os.environ.get("HF_TOKEN"),
    )
    if getattr(loaded, "safety_checker", None) is not None:
        loaded.safety_checker = None
    if hasattr(loaded, "requires_safety_checker"):
        loaded.requires_safety_checker = False
    if getattr(loaded, "watermarker", None) is not None:
        loaded.watermarker = None
    try:
        loaded.load_lora_weights(lora_path)
    except ImportError as err:
        print("load_lora_weights ImportError:", err)
        _purge_old_torchao()
        loaded.load_lora_weights(lora_path)
    try:
        loaded.unfuse_lora()
    except Exception:
        pass
    loaded.enable_model_cpu_offload()
    pipe = loaded
    print("LoRA loaded.")
    return pipe


def ensure_flux_dual_pipe():
    # v2 stays adapter 'default'. Male file loads as adapter 'hrmale'. Cells 13-40 keep default only.
    ensure_flux_pipe()
    male_path = os.path.join(LORAS_DIR, HENRY_OUTPUT_LORA)
    if not os.path.isfile(male_path):
        local_final = os.path.join(TRAIN_OUTPUT_DIR, HENRY_LORA_NAME, HENRY_OUTPUT_LORA)
        if os.path.isfile(local_final):
            male_path = local_final
    if not os.path.isfile(male_path):
        raise RuntimeError(
            "Male LoRA not found: " + male_path +
            " Run cells 41-45 first. Do not retrain lapetitemilf_flux_v2."
        )
    if getattr(pipe, "_henry_adapter_loaded", False):
        print("Male adapter already on this pipe:", male_path)
        return pipe
    print("Loading second LoRA (male, adapter hrmale):", male_path)
    try:
        pipe.load_lora_weights(male_path, adapter_name="hrmale")
    except TypeError as err:
        raise RuntimeError(
            "This diffusers build cannot stack two LoRAs (%s). "
            "Runtime > Restart session, rerun cells 1-4, then 46." % err
        )
    pipe._henry_adapter_loaded = True
    print("Stacked LoRAs: default=%s + hrmale=%s" % (OUTPUT_LORA_NAME, HENRY_OUTPUT_LORA))
    return pipe


def write_henry_yaml(path, steps, dry):
    sample_flag = "true" if dry else "false"
    skip_first = "true"
    save_every = 10000 if dry else SAVE_EVERY
    sample_every = 10000 if dry else SAMPLE_EVERY
    lines = [
        "job: extension",
        "config:",
        '  name: "%s"' % HENRY_LORA_NAME,
        "  process:",
        "    - type: sd_trainer",
        '      training_folder: "%s"' % TRAIN_OUTPUT_DIR,
        "      device: cuda:0",
        '      trigger_word: "%s"' % HENRY_TRIGGER,
        "      network:",
        "        type: lora",
        "        linear: %d" % NETWORK_DIM,
        "        linear_alpha: %d" % NETWORK_ALPHA,
        "      save:",
        "        dtype: float16",
        "        save_every: %d" % save_every,
        "        max_step_saves_to_keep: 4",
        "        push_to_hub: false",
        "      datasets:",
        '        - folder_path: "%s"' % HENRY_DATASET_DIR,
        "          caption_ext: txt",
        "          caption_dropout_rate: 0.05",
        "          shuffle_tokens: false",
        "          cache_latents_to_disk: true",
        "          resolution: [512, 768, 1024]",
        "      train:",
        "        batch_size: 1",
        "        steps: %d" % steps,
        "        gradient_accumulation_steps: 1",
        "        train_unet: true",
        "        train_text_encoder: false",
        "        gradient_checkpointing: true",
        "        noise_scheduler: flowmatch",
        "        optimizer: adamw8bit",
        "        lr: %.0e" % LEARNING_RATE,
        "        skip_first_sample: %s" % skip_first,
        "        disable_sampling: %s" % sample_flag,
        "        ema_config:",
        "          use_ema: true",
        "          ema_decay: 0.99",
        "        dtype: bf16",
        "      model:",
        '        name_or_path: "black-forest-labs/FLUX.1-dev"',
        "        is_flux: true",
        "        quantize: true",
        "      sample:",
        "        sampler: flowmatch",
        "        sample_every: %d" % sample_every,
        "        sample_start_step: 0",
        "        width: 1024",
        "        height: 1024",
        "        prompts:",
        '          - "hrmale, erect penis close-up, visible glans, veined shaft, photorealistic raw photo"',
        '          - "hrmale, erect penis, glans and shaft veins, waist-down close-up, photorealistic photo"',
        '          - "hrmale, side view erect penis, glans, shaft, natural skin texture, photorealistic"',
        '          - "hrmale, looking down at an erect penis, glans and veins, photorealistic raw photo"',
        '        neg: ""',
        "        seed: 42",
        "        walk_seed: true",
        "        guidance_scale: 4",
        "        sample_steps: 20",
        "meta:",
        '  name: "[name]"',
        '  version: "1.0"',
        "",
    ]
    text = chr(10).join(lines)
    if any(ord(ch) > 127 for ch in text):
        raise RuntimeError("YAML is not ASCII")
    if LORA_NAME in text or OUTPUT_LORA_NAME in text:
        raise RuntimeError("Henry YAML must not name the locked v2 LoRA.")
    with open(path, "w", encoding="ascii") as fh:
        fh.write(text)
    print("Wrote", path, "steps=%d dry=%s name=%s" % (steps, dry, HENRY_LORA_NAME))


def set_pipe_adapters(pipe, names, weights):
    names = list(names)
    weights = list(weights)
    try:
        pipe.set_adapters(names, adapter_weights=weights)
        return names
    except (ValueError, KeyError) as err:
        alt = ["default_0" if n == "default" else n for n in names]
        if alt == names:
            print("set_adapters:", err)
            raise
        print("set_adapters: default missing, using default_0")
        pipe.set_adapters(alt, adapter_weights=weights)
        return alt


def run_far_strip(slug, place, shots, seed_base, shot_start=0, shot_end=20):
    import os
    import torch
    from datetime import datetime
    from IPython.display import display
    if not SUBJECT_IS_ADULT:
        raise RuntimeError("Adult subject only.")
    if len(shots) != 20:
        raise RuntimeError("Need 20 shots in this series. Got %d" % len(shots))
    if shot_start < 0 or shot_end > 20 or shot_start >= shot_end:
        raise RuntimeError("SHOT_START/END must be inside 0..20 and START < END")
    ensure_flux_pipe()
    far = (
        "full body from a far camera, she is small in the frame, "
        "the location fills most of the shot"
    )
    ident = (
        "ohwx woman, an adult woman with long highlighted blonde hair and brown eyes, "
        "fair pale skin, natural soft teardrop breasts, medium circular pinkish-tan textured areolae, prominent nipples, "
    )
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = os.path.join(EVAL_DIR, "strip_" + slug + "_" + stamp)
    os.makedirs(out_dir, exist_ok=True)
    print("Series", slug)
    print("Shots", shot_start, "to", shot_end, "->", out_dir)
    print("Do not write scars or surgical in these prompts.")
    print("Keep this tab open.")
    saved = []
    for pidx in range(shot_start, shot_end):
        shot_slug, kind, action = shots[pidx]
        if kind == "nude":
            weight = 0.75
            guidance = 2.5
        else:
            weight = 1.0
            guidance = 3.5
        set_pipe_adapters(pipe, ["default"], [weight])
        seed = seed_base + pidx
        prompt = (
            ident + far + ", " + action + ". " + place +
            ", photorealistic raw photo, natural skin texture"
        )
        print("---", shot_slug, "seed", seed, kind)
        print(prompt)
        image = pipe(
            prompt=prompt,
            guidance_scale=guidance,
            height=1024,
            width=768,
            num_inference_steps=32,
            generator=torch.Generator("cuda").manual_seed(seed),
        ).images[0]
        path = os.path.join(out_dir, "%s_seed%d.png" % (shot_slug, seed))
        image.save(path)
        saved.append(path)
        print("saved", path)
        display(image)
    print("Saved", len(saved), "pictures in", out_dir)
    if USE_DRIVE_API:
        for path in saved:
            upload_project_file(path, os.path.relpath(path, ROOT))
    print("If it stopped early, set SHOT_START to the next index and rerun this cell.")
    print("Copy keepers to MyDrive/FiratSuper/keepers/")
    print("Do not put these pictures back into the training folders.")


def run_scene_set(
    slug,
    place,
    shots,
    seed_base,
    shot_start=0,
    shot_end=20,
    ident=None,
    adapter_names=None,
    adapter_weights=None,
    height=1024,
    width=768,
):
    import os
    import torch
    from datetime import datetime
    from IPython.display import display
    if not SUBJECT_IS_ADULT:
        raise RuntimeError("Adult subject only.")
    nshots = len(shots)
    if nshots < 1:
        raise RuntimeError("Need at least 1 shot in this series.")
    if shot_start < 0 or shot_end > nshots or shot_start >= shot_end:
        raise RuntimeError(
            "SHOT_START/END must be inside 0..%d and START < END" % nshots
        )
    use_male = adapter_names is not None and "hrmale" in adapter_names
    if use_male:
        ensure_flux_dual_pipe()
    else:
        ensure_flux_pipe()
    if ident is None:
        ident = (
            "ohwx woman, an adult woman with long highlighted blonde hair and brown eyes, "
            "fair pale skin, natural soft teardrop breasts, medium circular pinkish-tan textured areolae, prominent nipples, "
        )
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = os.path.join(EVAL_DIR, "scene_" + slug + "_" + stamp)
    os.makedirs(out_dir, exist_ok=True)
    print("Scene set", slug)
    print("Shots", shot_start, "to", shot_end, "->", out_dir)
    if use_male:
        print("Adapters:", adapter_names, adapter_weights)
        print("Prompts lead with genital/facial words (CLIP 77).")
    else:
        print("Two-person shots often glitch on Flux. That is the base model, not a bad LoRA.")
    print("Do not write scars or surgical in these prompts.")
    print("Keep this tab open.")
    saved = []
    for pidx in range(shot_start, shot_end):
        shot_slug, kind, action = shots[pidx]
        if kind in ("nude", "sex"):
            weight = 0.7
            guidance = 2.5
        else:
            weight = 0.9
            guidance = 3.0
        if adapter_names is None:
            names = ["default"]
            weights = [weight]
        else:
            names = list(adapter_names)
            if adapter_weights is None:
                weights = [weight] * len(names)
            else:
                weights = list(adapter_weights)
        names = set_pipe_adapters(pipe, names, weights)
        seed = seed_base + pidx
        if ident:
            prompt = ident + action + ". " + place + ", photorealistic raw photo, natural skin texture"
        else:
            prompt = action + ". " + place + ", photorealistic raw photo, natural skin texture"
        print("---", shot_slug, "seed", seed, kind, "lora", names, weights)
        print(prompt)
        image = pipe(
            prompt=prompt,
            guidance_scale=guidance,
            height=height,
            width=width,
            num_inference_steps=32,
            generator=torch.Generator("cuda").manual_seed(seed),
        ).images[0]
        path = os.path.join(out_dir, "%s_seed%d.png" % (shot_slug, seed))
        image.save(path)
        saved.append(path)
        print("saved", path)
        display(image)
    print("Saved", len(saved), "pictures in", out_dir)
    if USE_DRIVE_API:
        for path in saved:
            upload_project_file(path, os.path.relpath(path, ROOT))
    print("If it stopped early, set SHOT_START to the next index and rerun this cell.")
    print("Copy keepers to MyDrive/FiratSuper/keepers/")
    print("Do not put these pictures back into the training folders.")


print("Drive settings OK. Helpers ready for step2 generate (cell 5).")

In [ ]:
# @title 3) Hugging Face login (FLUX.1-dev is gated)
import getpass
import os

print("1. Open https://huggingface.co/black-forest-labs/FLUX.1-dev")
print("2. Accept the license while logged in")
print("3. Create a READ token: https://huggingface.co/settings/tokens")
print("4. Paste it below. Colab will hide it.")

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        print("Using HF_TOKEN from Colab Secrets.")
except Exception:
    token = None

if isinstance(token, dict):
    picked = token.get("value")
    if not picked:
        for item in token.values():
            if isinstance(item, str) and item:
                picked = item
                break
    token = picked
token = str(token).strip() if token else token
if not token:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if isinstance(token, dict):
        picked = token.get("value")
        if not picked:
            for item in token.values():
                if isinstance(item, str) and item:
                    picked = item
                    break
        token = picked
    token = str(token).strip() if token else token
if not token:
    token = getpass.getpass("HF READ token: ").strip()
if not token:
    raise RuntimeError("No Hugging Face token. Paste a READ token and rerun.")

os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import HfApi, login
login(token=token, add_to_git_credential=False)
api = HfApi(token=token)
try:
    api.model_info("black-forest-labs/FLUX.1-dev")
except Exception as err:
    raise RuntimeError(
        "Cannot read black-forest-labs/FLUX.1-dev. "
        "Accept the license on the model page, use a READ token from the same account. "
        "Detail: " + str(err)
    )
print("Hugging Face login OK. FLUX.1-dev is readable.")

In [ ]:
# @title 4) Install Ostris ai-toolkit (Gate 3)
import os
import sys
import subprocess
import shutil

INSTALL_MARK = "/content/ai-toolkit/.firat_install_ok"
INSTALL_VER = "2"
ROOT_TK = "/content/ai-toolkit"
SKIP_PKGS = ("torchcodec", "av==", "librosa==", "mutagen==", "gradio")


def run(cmd, cwd=None):
    print("+", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    ret = subprocess.call(cmd, cwd=cwd, env=env)
    if ret != 0:
        raise RuntimeError("command failed (%d): %s" % (ret, " ".join(cmd)))


def pip_req(path):
    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--prefer-binary",
            "--only-binary=scipy,numpy",
            "-r",
            path,
        ],
        cwd=ROOT_TK,
    )


def write_slim(src, dest):
    scipy_line = "scipy>=1.14.1\n" if sys.version_info >= (3, 13) else "scipy>=1.12.0\n"
    out = [scipy_line]
    with open(src, encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                out.append(line if line.endswith("\n") else line + "\n")
                continue
            if any(stripped.startswith(s) for s in SKIP_PKGS):
                print("skip", stripped)
                continue
            out.append(line if line.endswith("\n") else line + "\n")
    with open(dest, "w", encoding="utf-8") as fh:
        fh.writelines(out)
    print("Wrote", dest)


print("Python", sys.version.replace("\n", " "))
print("This cell can take several minutes. Let it finish.")

if not os.path.isdir(os.path.join(ROOT_TK, ".git")):
    if os.path.isdir(ROOT_TK):
        shutil.rmtree(ROOT_TK)
    run(["git", "clone", "https://github.com/ostris/ai-toolkit", ROOT_TK])
run(["git", "submodule", "update", "--init", "--recursive"], cwd=ROOT_TK)

# Ostris pins scipy==1.12.0. No Python 3.13 wheel, pip tries to compile and dies.
req_overlay = os.path.join(ROOT_TK, "requirements_colab.txt")
if sys.version_info >= (3, 13):
    with open(req_overlay, "w", encoding="ascii") as fh:
        fh.write("-r requirements_base.txt\n")
        fh.write("scipy>=1.14.1\n")
    print("Python 3.13+: using scipy>=1.14.1 instead of scipy==1.12.0")
else:
    req_overlay = os.path.join(ROOT_TK, "requirements.txt")

already = False
if os.path.isfile(INSTALL_MARK):
    already = open(INSTALL_MARK, encoding="ascii").read().strip() == INSTALL_VER

if already:
    print("ai-toolkit packages already installed (mark %s). Skipping pip." % INSTALL_VER)
else:
    run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "hf_transfer"])
    print("Installing Ostris requirements. If it fails, the red pip error is above this traceback.")
    try:
        pip_req(req_overlay)
    except RuntimeError:
        print("Full requirements failed. Retry without video extras (not needed for image LoRA).")
        slim = os.path.join(ROOT_TK, "requirements_colab_slim.txt")
        write_slim(os.path.join(ROOT_TK, "requirements_base.txt"), slim)
        pip_req(slim)
    with open(INSTALL_MARK, "w", encoding="ascii") as fh:
        fh.write(INSTALL_VER)
    print("Wrote", INSTALL_MARK)

sys.path.insert(0, ROOT_TK)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("ai-toolkit clone:", ROOT_TK)
print("If Colab asks to Restart Runtime, restart, then rerun cells 1-4.")
print("Gate 3 install OK.")

# step2 Cell 5 — woman-only face lock + photoreal (no couple, no paste)

**step2 תא 5 — אישה בלבד, נעילת פנים + פוטוריאליסטי (בלי זוג, בלי הדבקה)**

Mirrors **cell 77** in `Flux_LoRA_Training_Colab.ipynb` (approved clone of cell 68): woman-only, `lapetitemilf_flux_v2` @ **1.15**, prompt-only keeper names (no IP-Adapter / Redux — cell 77 had none). Face: `01_face_ok`–`04_face_ok` + `02_stand_three_q_chests` + approved scene81 prepaste names `17_scene81_seed8002_prepaste_ok` / `18_scene81_seed8101_prepaste_ok` in `keepers/output`. Chest is the 77 **text** recipe (`chest_real`). **No hard paste, no img2img, no compositing.** Photoreal raw photo; cartoon / illustration / anime / 3D / plastic in the negative.

Do **not** shorten like old cells 73/82 (that broke identity). Identity + face + chest tokens stay FIRST. CLIP length is printed as info only, same as cell 77. No man, no male LoRA, no couple.

**Run:** A100. Cells **1 → 2 → 3 → 4** then **Runtime → Run this cell only.**

~12 woman-only stills. Seeds **8002** and **8101** (anchors) plus **8300–8309**. Mild waist-up lace/sheer variety.
Writes `MyDrive/FiratSuper/generate/scene_step2_05_woman_face_photoreal_<timestamp>/`.

### עברית
A100. תאים **1 → 2 → 3 → 4**. אחר כך **Runtime → Run this cell only** (רק תא 5). אישה בלבד, פנים כמו תא 77, פוטוריאליסטי, בלי הדבקה ובלי זוג. בלי אימון.


In [ ]:
# @title 5) Woman-only face lock + photoreal (no couple, no paste)
# Generate-only. Do NOT train. Do NOT overwrite .safetensors.
# step2 Cell 5 (was going to be cell 83 in the long notebook — not added there).
# Cell 82 REJECTED: wrong face, cartoonish, bad chest. Do not iterate 82.
# MIRROR cell 77 / 68 identity recipe. Prompt-only keeper names.
# No IP-Adapter / Redux / img2img / hard paste / chest_real compositing.
# No man, no hrmale, no couple. v2 @ 1.15 only.
# Do NOT shorten prompts (cell 73/82 broke identity). Identity FIRST.
# CLIP length is info only, same as cell 77.
# Writes MyDrive/FiratSuper/generate/scene_step2_05_woman_face_photoreal_<timestamp>/
import os
import torch
from datetime import datetime
from IPython.display import display

FACE_OK_SCENE66 = "1QZMUC79DFg3kPPLT51lES-Xfml0zTPIs"
SHOT_START = 0
SHOT_END = 12
SLUG = "step2_05_woman_face_photoreal"
# Anchors 8002 + 8101 (approved scene81 prepaste) plus a new 8300-8309 set.
SEEDS = [8002, 8101, 8300, 8301, 8302, 8303, 8304, 8305, 8306, 8307, 8308, 8309]
LORA_W = 1.15
NEG = (
    "cartoon, illustration, anime, 3d render, cgi, painting, digital art, "
    "plastic skin, beauty filter, over-smoothed, airbrushed, "
    "glasses, gold necklace, black tank, gold curtains, "
    "chin crop, missing top of head, hrmale, man, couple, penis, "
    "multiple women, triplets, clone army, duplicate person"
)
# Cell 77/68 prompt blocks -- do not shorten. Identity + face + chest FIRST.
IDENT = (
    "ohwx woman, adult woman, long highlighted blonde hair, brown eyes, "
    "head fully in frame, "
)
FACE = (
    "matching the face identity of keeper stills "
    "01_face_ok, 02_face_ok, 03_face_ok, 04_face_ok, "
    "and the accepted face in 02_stand_three_q_chests, "
    "17_scene81_seed8002_prepaste_ok, 18_scene81_seed8101_prepaste_ok, "
)
CHEST = (
    "fair pale skin, slim torso, natural teardrop hang, "
    "medium circular pinkish-tan Montgomery-textured areolae, "
    "prominent nipples, matching chest_real, "
)
PLACE = (
    "full head in frame, space above the hair, waist-up medium shot, "
    "intimate bedroom, looking at the camera, "
    "photorealistic raw photo, natural skin texture, real camera photograph"
)
# Mild pose/clothing variety. Framing stays waist-up face+chest like cell 77.
SHOTS = [
    ("01_waist_pale_lace", "scene", "standing waist-up in soft pale lace lingerie, chest readable"),
    ("02_waist_pale_lace", "scene", "standing waist-up in soft pale lace lingerie, chest readable"),
    ("03_waist_sheer_pale", "scene", "standing waist-up in sheer pale lingerie, chest readable"),
    ("04_waist_pale_lace", "scene", "standing waist-up in pale lace lingerie, chest readable"),
    ("05_three_q_pale_lace", "scene", "three-quarter waist-up in pale lace lingerie, chest readable"),
    ("06_waist_sheer_pale", "scene", "standing waist-up in sheer pale lingerie, looking at the camera"),
    ("07_waist_pale_lace", "scene", "standing waist-up in soft pale lace lingerie, looking at the camera"),
    ("08_three_q_sheer", "scene", "three-quarter waist-up in sheer pale lingerie, chest readable"),
    ("09_waist_pale_lace", "scene", "standing waist-up in pale lace lingerie, indoor daylight"),
    ("10_waist_sheer_pale", "scene", "standing waist-up in sheer pale lingerie, chest readable"),
    ("11_three_q_pale_lace", "scene", "three-quarter waist-up in soft pale lace lingerie, chest readable"),
    ("12_waist_pale_lace", "scene", "standing waist-up in soft pale lace lingerie, chest readable"),
]
# Substring bans. Do NOT include "man" here -- it matches inside "woman" (cell 77 style).
BANNED = (
    "hrmale", "penis", "glans", "semen", "couple",
    "glasses", "gold necklace", "black tank", "gold curtains", "chin crop",
    "cartoon", "anime", "illustration",
)
KEEPER_STEER_NAMES = [
    "01_face_ok.png",
    "02_face_ok.png",
    "03_face_ok.png",
    "04_face_ok.png",
    "17_scene81_seed8002_prepaste_ok.png",
    "18_scene81_seed8101_prepaste_ok.png",
]

if SHOT_END != 12 or len(SHOTS) != 12 or len(SEEDS) != 12:
    raise RuntimeError("step2 Cell 5 must be exactly 12 woman-only stills.")
if 8002 not in SEEDS or 8101 not in SEEDS:
    raise RuntimeError("step2 Cell 5 must include anchor seeds 8002 and 8101.")
if not SUBJECT_IS_ADULT:
    raise RuntimeError("Adult subject only.")
if abs(LORA_W - 1.15) > 1e-6:
    raise RuntimeError("step2 Cell 5 must keep v2 at 1.15 (clone of cell 77/68).")

v2_path = os.path.join(LORAS_DIR, OUTPUT_LORA_NAME)
if OUTPUT_LORA_NAME != "lapetitemilf_flux_v2.safetensors":
    raise RuntimeError("step2 Cell 5 must load locked lapetitemilf_flux_v2 only.")
if not os.path.isfile(v2_path):
    raise RuntimeError("Load-only: missing " + v2_path + " (will not train).")
print("LOAD ONLY", OUTPUT_LORA_NAME, "bytes", os.path.getsize(v2_path))
v2_mtime, v2_size = os.path.getmtime(v2_path), os.path.getsize(v2_path)
print("MIRROR of cell 77. Woman-only. v2 @ 1.15. Prompt-only keepers. No IP-Adapter.")
print("Cell 82 REJECTED. Do not load male LoRA. No paste. No img2img.")
print("Prompt-only steer (folder %s). Not a dataset." % FACE_OK_SCENE66)

found_keepers = {}
search_roots = [
    os.path.join(KEEPERS_DIR, "output"),
    KEEPERS_DIR,
    os.path.join(ROOT, "keepers", "output"),
]
for root in search_roots:
    if not os.path.isdir(root):
        continue
    for dirpath, _dirs, files in os.walk(root):
        for name in KEEPER_STEER_NAMES:
            if name in files and name not in found_keepers:
                found_keepers[name] = os.path.join(dirpath, name)
print("Face keepers found (prompt steer, not loaded as IP-Adapter):", found_keepers or "(names only, like cell 77)")

ensure_flux_pipe()
used = set_pipe_adapters(pipe, ["default"], [LORA_W])
print("Female-only v2. Adapter:", used, "weight", LORA_W)
print("Male LoRA not loaded for this cell.")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
gen_parent = os.path.join(ROOT, "generate")
parts = gen_parent.split(os.sep)
if "keepers" in parts or "loras" in parts or any(p.startswith("ADD_") for p in parts):
    raise RuntimeError("Refusing to write under keepers/loras/ADD_*")
os.makedirs(gen_parent, exist_ok=True)
out_dir = os.path.join(gen_parent, "scene_" + SLUG + "_" + stamp)
os.makedirs(out_dir, exist_ok=True)
print("Out dir:", out_dir)
print("Drive path: MyDrive/FiratSuper/generate/" + os.path.basename(out_dir))

saved = []
for pidx in range(SHOT_START, SHOT_END):
    shot_slug, kind, action = SHOTS[pidx]
    used = set_pipe_adapters(pipe, ["default"], [LORA_W])
    prompt = IDENT + FACE + CHEST + action + ". " + PLACE
    low = prompt.lower()
    hit = [w for w in BANNED if w in low]
    if hit:
        raise RuntimeError("step2 Cell 5 prompt has banned words: " + ", ".join(hit))
    words = [w.strip(".,;:") for w in low.split()]
    if "man" in words or "couple" in words or "hrmale" in words:
        raise RuntimeError("step2 Cell 5 must stay woman-only.")
    for lock in (
        "ohwx woman",
        "long highlighted blonde hair",
        "brown eyes",
        "01_face_ok",
        "02_face_ok",
        "03_face_ok",
        "04_face_ok",
        "02_stand_three_q_chests",
        "17_scene81_seed8002_prepaste_ok",
        "18_scene81_seed8101_prepaste_ok",
        "fair pale skin",
        "slim torso",
        "natural teardrop hang",
        "medium circular pinkish-tan Montgomery-textured areolae",
        "prominent nipples",
        "chest_real",
        "full head in frame",
        "space above the hair",
        "waist-up medium shot",
        "photorealistic raw photo",
        "natural skin texture",
    ):
        if lock not in prompt and lock not in low:
            raise RuntimeError("step2 Cell 5 prompt missing lock: " + lock)
    clip_tok = getattr(pipe, "tokenizer", None)
    if clip_tok is not None:
        clip_n = len(clip_tok(prompt, add_special_tokens=True).input_ids)
        print("CLIP tokens", clip_n, "(info only; cell 77/68 length is intentional; do not shorten)")
    seed = SEEDS[pidx]
    fname = "%s_seed%d.png" % (shot_slug, seed)
    print("---", fname, "seed", seed, "lora", used, LORA_W)
    print(prompt)
    image = pipe(
        prompt=prompt,
        negative_prompt=NEG,
        guidance_scale=3.5,
        height=768,
        width=1024,
        num_inference_steps=32,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]
    path = os.path.join(out_dir, fname)
    image.save(path)
    saved.append(path)
    print("saved", path)
    display(image)

if os.path.getmtime(v2_path) != v2_mtime or os.path.getsize(v2_path) != v2_size:
    raise RuntimeError("v2 LoRA file changed during generate. Stop.")
print("Saved", len(saved), "stills in", out_dir)
print("Drive path: MyDrive/FiratSuper/generate/" + os.path.basename(out_dir))
if USE_DRIVE_API:
    for path in saved:
        if path.endswith(".safetensors"):
            raise RuntimeError("Refusing to upload safetensors from step2 Cell 5.")
        upload_project_file(path, os.path.relpath(path, ROOT))
fid = None
try:
    service = DRIVE_SERVICE or _api_service()
    gen_folder = api_ensure_folder(service, FIRATSUPER_DRIVE_ID, "generate")
    found = api_find_child(service, gen_folder, os.path.basename(out_dir))
    fid = found["id"] if found else gen_folder
    print("Drive folder id:", fid)
except Exception as err:
    print("Drive folder id lookup skipped:", err)
print("SCENE_STEP2_05_DIR", out_dir)
print("step2 Cell 5 done. Woman-only. LoRA files were not written. Male LoRA was not loaded.")
print("Do not put these pictures back into ADD_* or training folders.")


# step2 Cell 6 — woman-only bare chest QA (face lock from 5, no bra, no paste)

**step2 תא 6 — אישה בלבד, QA חזה חשוף (נעילת פנים מתא 5, בלי חזייה, בלי הדבקה)**

Cell 5 faces are OK (identity lock holds). Lingerie/bra made nipples sit on top of / poke through fabric (Flux sheer-bra artifact). This cell keeps the **same face lock** as cell 5 / cell 77 and asks for **topless / bare chest** so nipples can be QA'd without a bra in the way.

Woman-only, `lapetitemilf_flux_v2` @ **1.15**, prompt-only keeper names (no IP-Adapter / Redux). Face: `01_face_ok`–`04_face_ok` + `02_stand_three_q_chests` + scene81 prepaste `17_scene81_seed8002_prepaste_ok` / `18_scene81_seed8101_prepaste_ok`. Chest is the 77 / 81 **prepaste text** recipe (`chest_real`). **No hard paste, no img2img, no compositing.** No man, no male LoRA, no couple.

Photoreal raw photo; cartoon / illustration / anime / 3D / plastic in the negative. Extra clothing negatives: bra, bikini top, lace covering nipples, underwire, nipples through fabric, floating nipples on clothing.

Do **not** shorten like old cells 73/82. Identity + face + chest tokens stay FIRST. CLIP length is info only.

**Run:** A100. Cells **1 → 2 → 3 → 4** then **Runtime → Run this cell only.**

~12 woman-only stills. Seeds **8002** and **8101** (anchors) plus **8600–8609**. Mild waist-up variety; face + bare chest readable.
Writes `MyDrive/FiratSuper/generate/scene_step2_06_woman_bare_chest_<timestamp>/`.

### עברית
A100. תאים **1 → 2 → 3 → 4**. אחר כך **Runtime → Run this cell only** (רק תא 6). אישה בלבד, פנים כמו תא 5 / 77, חזה חשוף בלי חזייה, פוטוריאליסטי, בלי הדבקה ובלי זוג. בלי אימון.


In [ ]:
# @title 6) Woman-only bare chest QA (face lock from 5, no bra, no paste)
# Generate-only. Do NOT train. Do NOT overwrite .safetensors.
# step2 Cell 6: same face lock as cell 5 / 77. Topless / bare chest.
# Cell 5 faces OK; lingerie caused Flux sheer-bra (nipples on/through fabric).
# No IP-Adapter / Redux / img2img / hard paste / chest_real compositing.
# No man, no hrmale, no couple. v2 @ 1.15 only.
# Do NOT shorten prompts (cell 73/82 broke identity). Identity FIRST.
# CLIP length is info only, same as cell 77 / cell 5.
# Writes MyDrive/FiratSuper/generate/scene_step2_06_woman_bare_chest_<timestamp>/
import os
import torch
from datetime import datetime
from IPython.display import display

FACE_OK_SCENE66 = "1QZMUC79DFg3kPPLT51lES-Xfml0zTPIs"
SHOT_START = 0
SHOT_END = 12
SLUG = "step2_06_woman_bare_chest"
# Anchors 8002 + 8101 (approved scene81 prepaste) plus a new 8600-8609 set.
SEEDS = [8002, 8101, 8600, 8601, 8602, 8603, 8604, 8605, 8606, 8607, 8608, 8609]
LORA_W = 1.15
NEG = (
    "cartoon, illustration, anime, 3d render, cgi, painting, digital art, "
    "plastic skin, beauty filter, over-smoothed, airbrushed, "
    "bra, bikini top, lingerie, lace covering nipples, underwire, "
    "nipples through fabric, floating nipples on clothing, sheer bra, "
    "glasses, gold necklace, black tank, gold curtains, "
    "chin crop, missing top of head, hrmale, man, couple, penis, "
    "multiple women, triplets, clone army, duplicate person"
)
# Cell 77 / cell 5 prompt blocks -- do not shorten. Identity + face + chest FIRST.
IDENT = (
    "ohwx woman, adult woman, long highlighted blonde hair, brown eyes, "
    "head fully in frame, "
)
FACE = (
    "matching the face identity of keeper stills "
    "01_face_ok, 02_face_ok, 03_face_ok, 04_face_ok, "
    "and the accepted face in 02_stand_three_q_chests, "
    "17_scene81_seed8002_prepaste_ok, 18_scene81_seed8101_prepaste_ok, "
)
CHEST = (
    "fair pale skin, slim torso, natural teardrop hang, "
    "medium circular pinkish-tan Montgomery-textured areolae, "
    "prominent nipples, matching chest_real, "
)
PLACE = (
    "full head in frame, space above the hair, waist-up medium shot, "
    "topless, bare breasts, no bra, no lingerie on chest, "
    "intimate bedroom, looking at the camera, "
    "photorealistic raw photo, natural skin texture, real camera photograph"
)
# Mild pose variety. Framing stays waist-up face + bare chest. No bra / lace.
SHOTS = [
    ("01_waist_topless", "scene", "standing waist-up, topless, bare breasts, chest readable"),
    ("02_waist_topless", "scene", "standing waist-up, topless, bare chest, no bra"),
    ("03_three_q_topless", "scene", "three-quarter waist-up, topless, bare breasts, chest readable"),
    ("04_waist_topless", "scene", "standing waist-up, topless, bare breasts, looking at the camera"),
    ("05_waist_topless", "scene", "standing waist-up, topless, bare chest readable, indoor daylight"),
    ("06_three_q_topless", "scene", "three-quarter waist-up, topless, bare breasts, no bra"),
    ("07_waist_topless", "scene", "standing waist-up, topless, bare breasts, looking at the camera"),
    ("08_three_q_topless", "scene", "three-quarter waist-up, topless, bare chest, chest readable"),
    ("09_waist_topless", "scene", "standing waist-up, topless, bare breasts, indoor daylight"),
    ("10_waist_topless", "scene", "standing waist-up, topless, bare chest, no lingerie on chest"),
    ("11_three_q_topless", "scene", "three-quarter waist-up, topless, bare breasts, chest readable"),
    ("12_waist_topless", "scene", "standing waist-up, topless, bare breasts, chest readable"),
]
# Substring bans. Do NOT include "man" (matches woman) or "bra"/"lingerie"
# (positive prompt says "no bra" / "no lingerie on chest").
BANNED = (
    "hrmale", "penis", "glans", "semen", "couple",
    "glasses", "gold necklace", "black tank", "gold curtains", "chin crop",
    "cartoon", "anime", "illustration",
    "underwire", "bikini top",
)
KEEPER_STEER_NAMES = [
    "01_face_ok.png",
    "02_face_ok.png",
    "03_face_ok.png",
    "04_face_ok.png",
    "17_scene81_seed8002_prepaste_ok.png",
    "18_scene81_seed8101_prepaste_ok.png",
]

if SHOT_END != 12 or len(SHOTS) != 12 or len(SEEDS) != 12:
    raise RuntimeError("step2 Cell 6 must be exactly 12 woman-only stills.")
if 8002 not in SEEDS or 8101 not in SEEDS:
    raise RuntimeError("step2 Cell 6 must include anchor seeds 8002 and 8101.")
if not SUBJECT_IS_ADULT:
    raise RuntimeError("Adult subject only.")
if abs(LORA_W - 1.15) > 1e-6:
    raise RuntimeError("step2 Cell 6 must keep v2 at 1.15 (clone of cell 5 / 77).")

v2_path = os.path.join(LORAS_DIR, OUTPUT_LORA_NAME)
if OUTPUT_LORA_NAME != "lapetitemilf_flux_v2.safetensors":
    raise RuntimeError("step2 Cell 6 must load locked lapetitemilf_flux_v2 only.")
if not os.path.isfile(v2_path):
    raise RuntimeError("Load-only: missing " + v2_path + " (will not train).")
print("LOAD ONLY", OUTPUT_LORA_NAME, "bytes", os.path.getsize(v2_path))
v2_mtime, v2_size = os.path.getmtime(v2_path), os.path.getsize(v2_path)
print("MIRROR of cell 5 / 77 face lock. Woman-only. v2 @ 1.15. Topless / bare chest.")
print("Prompt-only keepers. No IP-Adapter. No male LoRA. No paste. No img2img.")
print("Prompt-only steer (folder %s). Not a dataset." % FACE_OK_SCENE66)

found_keepers = {}
search_roots = [
    os.path.join(KEEPERS_DIR, "output"),
    KEEPERS_DIR,
    os.path.join(ROOT, "keepers", "output"),
]
for root in search_roots:
    if not os.path.isdir(root):
        continue
    for dirpath, _dirs, files in os.walk(root):
        for name in KEEPER_STEER_NAMES:
            if name in files and name not in found_keepers:
                found_keepers[name] = os.path.join(dirpath, name)
print("Face keepers found (prompt steer, not loaded as IP-Adapter):", found_keepers or "(names only, like cell 77)")

ensure_flux_pipe()
used = set_pipe_adapters(pipe, ["default"], [LORA_W])
print("Female-only v2. Adapter:", used, "weight", LORA_W)
print("Male LoRA not loaded for this cell.")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
gen_parent = os.path.join(ROOT, "generate")
parts = gen_parent.split(os.sep)
if "keepers" in parts or "loras" in parts or any(p.startswith("ADD_") for p in parts):
    raise RuntimeError("Refusing to write under keepers/loras/ADD_*")
os.makedirs(gen_parent, exist_ok=True)
out_dir = os.path.join(gen_parent, "scene_" + SLUG + "_" + stamp)
os.makedirs(out_dir, exist_ok=True)
print("Out dir:", out_dir)
print("Drive path: MyDrive/FiratSuper/generate/" + os.path.basename(out_dir))

saved = []
for pidx in range(SHOT_START, SHOT_END):
    shot_slug, kind, action = SHOTS[pidx]
    used = set_pipe_adapters(pipe, ["default"], [LORA_W])
    prompt = IDENT + FACE + CHEST + action + ". " + PLACE
    low = prompt.lower()
    hit = [w for w in BANNED if w in low]
    if hit:
        raise RuntimeError("step2 Cell 6 prompt has banned words: " + ", ".join(hit))
    words = [w.strip(".,;:") for w in low.split()]
    if "man" in words or "couple" in words or "hrmale" in words:
        raise RuntimeError("step2 Cell 6 must stay woman-only.")
    for lock in (
        "ohwx woman",
        "long highlighted blonde hair",
        "brown eyes",
        "01_face_ok",
        "02_face_ok",
        "03_face_ok",
        "04_face_ok",
        "02_stand_three_q_chests",
        "17_scene81_seed8002_prepaste_ok",
        "18_scene81_seed8101_prepaste_ok",
        "fair pale skin",
        "slim torso",
        "natural teardrop hang",
        "medium circular pinkish-tan Montgomery-textured areolae",
        "prominent nipples",
        "chest_real",
        "topless",
        "bare breasts",
        "no bra",
        "no lingerie on chest",
        "full head in frame",
        "space above the hair",
        "waist-up medium shot",
        "photorealistic raw photo",
        "natural skin texture",
    ):
        if lock not in prompt and lock not in low:
            raise RuntimeError("step2 Cell 6 prompt missing lock: " + lock)
    clip_tok = getattr(pipe, "tokenizer", None)
    if clip_tok is not None:
        clip_n = len(clip_tok(prompt, add_special_tokens=True).input_ids)
        print("CLIP tokens", clip_n, "(info only; cell 77/68 length is intentional; do not shorten)")
    seed = SEEDS[pidx]
    fname = "%s_seed%d.png" % (shot_slug, seed)
    print("---", fname, "seed", seed, "lora", used, LORA_W)
    print(prompt)
    image = pipe(
        prompt=prompt,
        negative_prompt=NEG,
        guidance_scale=3.5,
        height=768,
        width=1024,
        num_inference_steps=32,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]
    path = os.path.join(out_dir, fname)
    image.save(path)
    saved.append(path)
    print("saved", path)
    display(image)

if os.path.getmtime(v2_path) != v2_mtime or os.path.getsize(v2_path) != v2_size:
    raise RuntimeError("v2 LoRA file changed during generate. Stop.")
print("Saved", len(saved), "stills in", out_dir)
print("Drive path: MyDrive/FiratSuper/generate/" + os.path.basename(out_dir))
if USE_DRIVE_API:
    for path in saved:
        if path.endswith(".safetensors"):
            raise RuntimeError("Refusing to upload safetensors from step2 Cell 6.")
        upload_project_file(path, os.path.relpath(path, ROOT))
fid = None
try:
    service = DRIVE_SERVICE or _api_service()
    gen_folder = api_ensure_folder(service, FIRATSUPER_DRIVE_ID, "generate")
    found = api_find_child(service, gen_folder, os.path.basename(out_dir))
    fid = found["id"] if found else gen_folder
    print("Drive folder id:", fid)
except Exception as err:
    print("Drive folder id lookup skipped:", err)
print("SCENE_STEP2_06_DIR", out_dir)
print("step2 Cell 6 done. Woman-only topless. LoRA files were not written. Male LoRA was not loaded.")
print("Do not put these pictures back into ADD_* or training folders.")


# step2 Cell 7 — weak img2img from approved keepers 17+18 (preserve chest)

**step2 תא 7 — img2img חלש משומרי 17+18 שאושרו (לשמור את החזה)**

Cell 6 REJECTED (nipples bad on every frame). Faces OK since cell 5. Text / hard paste / strong img2img did not lock `chest_real` nipples. This cell starts FROM Henry's approved keepers and only **weakly** varies them.

Load `keepers/output/17_scene81_seed8002_prepaste_ok.png` and `18_scene81_seed8101_prepaste_ok.png` (fail if missing). Woman-only, `lapetitemilf_flux_v2` @ **1.15**. **No hard-paste crops. No male LoRA. No bra.** Same face+chest text as cell 5/6. Photoreal. Topless like the keepers.

Denoise sweep **0.20 / 0.25 / 0.30** (band ~0.18–0.32). Print strength per file. ~12 stills.

**Run:** A100. Cells **1 → 2 → 3 → 4** then **Runtime → Run this cell only.**

Writes `MyDrive/FiratSuper/generate/scene_step2_07_from_keepers_17_18_<timestamp>/`.

### עברית
A100. תאים **1 → 2 → 3 → 4**. אחר כך **Runtime → Run this cell only** (רק תא 7). אישה בלבד. img2img חלש משני השומרים שאושרו. בלי הדבקת יבול, בלי חזייה, בלי אימון, בלי זוג.


In [ ]:
# @title 7) Weak img2img from approved keepers 17+18 (preserve chest)
# Generate-only. Do NOT train. Do NOT overwrite .safetensors.
# Cell 6 REJECTED (nipples). Faces OK since cell 5.
# Init images = the two approved FULL keepers. Weak denoise only.
# NOT cell 79 (that pasted chest_real crops then img2img @ 0.32).
# No hard-paste crops. No male LoRA. No bra.
# Writes MyDrive/FiratSuper/generate/scene_step2_07_from_keepers_17_18_<timestamp>/
import os
import gc
import inspect
import torch
from datetime import datetime
from PIL import Image
from IPython.display import display
from diffusers import FluxImg2ImgPipeline

FACE_OK_SCENE66 = "1QZMUC79DFg3kPPLT51lES-Xfml0zTPIs"
KEEPERS_OUTPUT_FOLDER_ID = "1McbLdO0usQC_IoaVDieGc0ubHO2wSLe5"
KEEPER_REFS = [
    ("17", "17_scene81_seed8002_prepaste_ok.png", "1HAF9MzdnM3rxs26JnCe7hEpEj_DfxFOr"),
    ("18", "18_scene81_seed8101_prepaste_ok.png", "1e13ASDzbMY2FbfirVH58ZZ6s437ydxeT"),
]
SLUG = "step2_07_from_keepers_17_18"
LORA_W = 1.15
WIDTH, HEIGHT = 1024, 768
IMG2IMG_STEPS = 28
GUIDANCE = 3.5
# Weak band ~0.18-0.32. Cell 79 @ 0.32 + crop paste warped nipples.
STRENGTHS = [0.20, 0.25, 0.30]
# 2 keepers x 3 strengths x 2 seeds = 12. Anchors 8002 / 8101 plus new 870x.
JOBS = [
    (1, "17", 0.20, 8002, "standing waist-up, topless, bare breasts, chest readable"),
    (2, "17", 0.25, 8002, "standing waist-up, topless, bare breasts, looking at the camera"),
    (3, "17", 0.30, 8002, "standing waist-up, topless, bare chest, no bra"),
    (4, "17", 0.20, 8700, "three-quarter waist-up, topless, bare breasts, chest readable"),
    (5, "17", 0.25, 8701, "standing waist-up, topless, bare breasts, indoor daylight"),
    (6, "17", 0.30, 8702, "three-quarter waist-up, topless, bare chest, no bra"),
    (7, "18", 0.20, 8101, "standing waist-up, topless, bare breasts, chest readable"),
    (8, "18", 0.25, 8101, "standing waist-up, topless, bare breasts, looking at the camera"),
    (9, "18", 0.30, 8101, "standing waist-up, topless, bare chest, no bra"),
    (10, "18", 0.20, 8710, "three-quarter waist-up, topless, bare breasts, chest readable"),
    (11, "18", 0.25, 8711, "standing waist-up, topless, bare breasts, indoor daylight"),
    (12, "18", 0.30, 8712, "three-quarter waist-up, topless, bare chest, no bra"),
]
NEG = (
    "cartoon, illustration, anime, 3d render, cgi, painting, digital art, "
    "plastic skin, beauty filter, over-smoothed, airbrushed, "
    "bra, bikini top, lingerie covering chest, lace covering nipples, underwire, "
    "nipples through fabric, floating nipples on clothing, sheer bra, "
    "warped nipples, misshapen nipples, stringy nipples, triangular nipples, "
    "glasses, gold necklace, black tank, gold curtains, "
    "chin crop, missing top of head, hrmale, man, couple, penis, "
    "multiple women, triplets, clone army, duplicate person"
)
IDENT = (
    "ohwx woman, adult woman, long highlighted blonde hair, brown eyes, "
    "head fully in frame, "
)
FACE = (
    "matching the face identity of keeper stills "
    "01_face_ok, 02_face_ok, 03_face_ok, 04_face_ok, "
    "and the accepted face in 02_stand_three_q_chests, "
    "17_scene81_seed8002_prepaste_ok, 18_scene81_seed8101_prepaste_ok, "
)
CHEST = (
    "fair pale skin, slim torso, natural teardrop hang, "
    "medium circular pinkish-tan Montgomery-textured areolae, "
    "prominent nipples, matching chest_real, "
)
PLACE = (
    "full head in frame, space above the hair, waist-up medium shot, "
    "topless, bare breasts, no bra, no lingerie on chest, "
    "intimate bedroom, looking at the camera, "
    "photorealistic raw photo, natural skin texture, real camera photograph"
)
BANNED = (
    "hrmale", "penis", "glans", "semen", "couple",
    "glasses", "gold necklace", "black tank", "gold curtains", "chin crop",
    "cartoon", "anime", "illustration",
    "underwire", "bikini top",
)


def _resample():
    try:
        return Image.Resampling.LANCZOS
    except AttributeError:
        return Image.LANCZOS


def load_approved_keepers():
    """Load the two approved FULL keepers. Local keepers/output first, else Drive ids."""
    cache_dir = "/tmp/firat_step2_keepers_17_18"
    os.makedirs(cache_dir, exist_ok=True)
    local_dirs = [
        os.path.join(KEEPERS_DIR, "output"),
        KEEPERS_DIR,
        os.path.join(ROOT, "keepers", "output"),
        os.path.join(ROOT, "keepers"),
    ]
    service = None

    def _service():
        nonlocal service
        if service is None:
            service = globals().get("DRIVE_SERVICE") or _api_service()
        return service

    folder_names = {}
    try:
        for item in api_list_children(_service(), KEEPERS_OUTPUT_FOLDER_ID):
            folder_names[item.get("name", "")] = item.get("id")
    except Exception as err:
        print("keepers/output folder list skipped:", err)

    loaded = {}
    for key, name, file_id in KEEPER_REFS:
        path = None
        for d in local_dirs:
            cand = os.path.join(d, name)
            if os.path.isfile(cand) and os.path.getsize(cand) > 1000:
                path = cand
                print("keeper local", cand)
                break
        if path is None:
            dest = os.path.join(cache_dir, name)
            if os.path.isfile(dest) and os.path.getsize(dest) > 1000:
                path = dest
                print("keeper cache", dest)
            else:
                use_id = file_id or folder_names.get(name)
                if not use_id:
                    raise RuntimeError(
                        "Missing Drive file id for %s. Expected keepers/output/%s "
                        "(folder %s)." % (name, name, KEEPERS_OUTPUT_FOLDER_ID)
                    )
                print("Downloading keeper", name, use_id)
                api_download_file(_service(), use_id, dest)
                if os.path.isfile(dest) and os.path.getsize(dest) > 1000:
                    path = dest
                    local_out = os.path.join(KEEPERS_DIR, "output")
                    os.makedirs(local_out, exist_ok=True)
                    local_copy = os.path.join(local_out, name)
                    if not os.path.isfile(local_copy):
                        try:
                            import shutil
                            shutil.copy2(dest, local_copy)
                            print("copied keeper to", local_copy)
                        except Exception as err:
                            print("keeper local copy skipped:", err)
        if path is None or not os.path.isfile(path) or os.path.getsize(path) < 1000:
            raise RuntimeError(
                "Approved keeper missing: %s. Expected "
                "/content/drive/MyDrive/FiratSuper/keepers/output/%s "
                "or Drive file id %s (folder %s)."
                % (name, name, file_id, KEEPERS_OUTPUT_FOLDER_ID)
            )
        img = Image.open(path).convert("RGB")
        init = img.resize((WIDTH, HEIGHT), _resample())
        loaded[key] = {"key": key, "name": name, "id": file_id, "path": path, "image": init}
        print("LOADED keeper", key, name, img.size, "->", init.size, os.path.getsize(path), "bytes")
        display(init)
    if set(loaded) != {"17", "18"}:
        raise RuntimeError("Cell 7 must load BOTH keepers 17 and 18. Got: " + str(sorted(loaded)))
    return loaded


def ensure_flux_img2img_wrap():
    """Cell 11 wrap: same Flux components, no from_pipe, no second download."""
    ensure_flux_pipe()
    if (
        "img2img_pipe" in globals()
        and img2img_pipe is not None
        and type(img2img_pipe).__name__ == "FluxImg2ImgPipeline"
    ):
        print("Using Flux img2img already wrapped (cell 11 pattern).")
        return img2img_pipe
    print("VRAM before offload: %.1f GB" % (torch.cuda.memory_allocated() / 1024**3))
    try:
        pipe.enable_model_cpu_offload()
    except Exception as err:
        print("enable_model_cpu_offload:", err)
    gc.collect()
    torch.cuda.empty_cache()
    sig = inspect.signature(FluxImg2ImgPipeline.__init__)
    comps = {}
    for key, value in pipe.components.items():
        if key in sig.parameters:
            comps[key] = value
    wrapped = FluxImg2ImgPipeline(**comps)
    print("Wrapped Flux components as img2img. No from_pipe, no second download. (cell 11)")
    try:
        wrapped.enable_model_cpu_offload()
    except Exception as err:
        print("img2img offload:", err)
    print("VRAM after wrap: %.1f GB" % (torch.cuda.memory_allocated() / 1024**3))
    return wrapped


if len(JOBS) != 12:
    raise RuntimeError("step2 Cell 7 must be exactly 12 stills.")
if any(s < 0.18 or s > 0.32 for s in STRENGTHS):
    raise RuntimeError("step2 Cell 7 denoise must stay in ~0.18-0.32.")
if any(job[2] < 0.18 or job[2] > 0.32 for job in JOBS):
    raise RuntimeError("step2 Cell 7 job strength outside weak band.")
if not SUBJECT_IS_ADULT:
    raise RuntimeError("Adult subject only.")
if abs(LORA_W - 1.15) > 1e-6:
    raise RuntimeError("step2 Cell 7 must keep v2 at 1.15 (clone of cell 5 / 77).")

v2_path = os.path.join(LORAS_DIR, OUTPUT_LORA_NAME)
if OUTPUT_LORA_NAME != "lapetitemilf_flux_v2.safetensors":
    raise RuntimeError("step2 Cell 7 must load locked lapetitemilf_flux_v2 only.")
if not os.path.isfile(v2_path):
    raise RuntimeError("Load-only: missing " + v2_path + " (will not train).")
print("LOAD ONLY", OUTPUT_LORA_NAME, "bytes", os.path.getsize(v2_path))
v2_mtime, v2_size = os.path.getmtime(v2_path), os.path.getsize(v2_path)
print("WEAK img2img from approved FULL keepers 17+18. No crop paste.")
print("Denoise sweep:", STRENGTHS, "band 0.18-0.32")
print("Prompt-only face names (folder %s). Chest pixels come from the keeper PNGs." % FACE_OK_SCENE66)

keepers = load_approved_keepers()
ensure_flux_pipe()
used = set_pipe_adapters(pipe, ["default"], [LORA_W])
print("Female-only v2. Adapter:", used, "weight", LORA_W)
print("Male LoRA not loaded for this cell.")
img2img_pipe = ensure_flux_img2img_wrap()
set_pipe_adapters(img2img_pipe, ["default"], [LORA_W])

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
gen_parent = os.path.join(ROOT, "generate")
parts = gen_parent.split(os.sep)
if "keepers" in parts or "loras" in parts or any(p.startswith("ADD_") for p in parts):
    raise RuntimeError("Refusing to write under keepers/loras/ADD_*")
os.makedirs(gen_parent, exist_ok=True)
out_dir = os.path.join(gen_parent, "scene_" + SLUG + "_" + stamp)
os.makedirs(out_dir, exist_ok=True)
print("Out dir:", out_dir)
print("Drive path: MyDrive/FiratSuper/generate/" + os.path.basename(out_dir))

for key, rec in keepers.items():
    src_name = "00_src_from%s.png" % key
    src_path = os.path.join(out_dir, src_name)
    rec["image"].save(src_path)
    print("saved source keeper copy", src_path)

saved = []
for n, key, strength, seed, action in JOBS:
    rec = keepers[key]
    used = set_pipe_adapters(img2img_pipe, ["default"], [LORA_W])
    prompt = IDENT + FACE + CHEST + action + ". " + PLACE
    low = prompt.lower()
    hit = [w for w in BANNED if w in low]
    if hit:
        raise RuntimeError("step2 Cell 7 prompt has banned words: " + ", ".join(hit))
    words = [w.strip(".,;:") for w in low.split()]
    if "man" in words or "couple" in words or "hrmale" in words:
        raise RuntimeError("step2 Cell 7 must stay woman-only.")
    for lock in (
        "ohwx woman",
        "long highlighted blonde hair",
        "brown eyes",
        "01_face_ok",
        "04_face_ok",
        "17_scene81_seed8002_prepaste_ok",
        "18_scene81_seed8101_prepaste_ok",
        "fair pale skin",
        "slim torso",
        "natural teardrop hang",
        "medium circular pinkish-tan Montgomery-textured areolae",
        "prominent nipples",
        "chest_real",
        "topless",
        "bare breasts",
        "no bra",
        "photorealistic raw photo",
    ):
        if lock not in prompt and lock not in low:
            raise RuntimeError("step2 Cell 7 prompt missing lock: " + lock)
    clip_tok = getattr(img2img_pipe, "tokenizer", None) or getattr(pipe, "tokenizer", None)
    if clip_tok is not None:
        clip_n = len(clip_tok(prompt, add_special_tokens=True).input_ids)
        print("CLIP tokens", clip_n, "(info only; cell 77/68 length is intentional; do not shorten)")
    str_tag = "str%03d" % int(round(strength * 100))
    fname = "%02d_from%s_%s_seed%d.png" % (n, key, str_tag, seed)
    print("---", fname, "denoise/strength", strength, "keeper", rec["name"], "seed", seed, "lora", used, LORA_W)
    print("init", rec["path"])
    print(prompt)
    image = img2img_pipe(
        prompt=prompt,
        negative_prompt=NEG,
        image=rec["image"],
        strength=strength,
        guidance_scale=GUIDANCE,
        num_inference_steps=IMG2IMG_STEPS,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]
    path = os.path.join(out_dir, fname)
    image.save(path)
    saved.append(path)
    print("saved", path, "denoise/strength", strength)
    display(image)
    gc.collect()
    torch.cuda.empty_cache()

if os.path.getmtime(v2_path) != v2_mtime or os.path.getsize(v2_path) != v2_size:
    raise RuntimeError("v2 LoRA file changed during generate. Stop.")
print("Saved", len(saved), "stills in", out_dir)
print("Drive path: MyDrive/FiratSuper/generate/" + os.path.basename(out_dir))
print("Denoise used:", ", ".join("%s=%.2f" % (os.path.basename(p), JOBS[i][2]) for i, p in enumerate(saved)))
if USE_DRIVE_API:
    for path in saved:
        if path.endswith(".safetensors"):
            raise RuntimeError("Refusing to upload safetensors from step2 Cell 7.")
        upload_project_file(path, os.path.relpath(path, ROOT))
fid = None
try:
    service = DRIVE_SERVICE or _api_service()
    gen_folder = api_ensure_folder(service, FIRATSUPER_DRIVE_ID, "generate")
    found = api_find_child(service, gen_folder, os.path.basename(out_dir))
    fid = found["id"] if found else gen_folder
    print("Drive folder id:", fid)
except Exception as err:
    print("Drive folder id lookup skipped:", err)
print("SCENE_STEP2_07_DIR", out_dir)
print("step2 Cell 7 done. Weak img2img from keepers 17+18. No crop paste. Male LoRA not loaded.")
print("Do not put these pictures back into ADD_* or training folders.")


# step2 Cell 8 — train small chest/nipple LoRA from chest_real only

**step2 תא 8 — אימון LoRA קטן לחזה/פטמות מ-chest_real בלבד**

Cell 7 REJECTED (weak img2img still unreal nipples). Faces OK since cell 5. Text / paste / strong+weak img2img all failed. This cell trains a **small chest adapter from REAL photos only**.

**Does NOT touch** locked `lapetitemilf_flux_v2`. New file: `loras/lapetitemilf_chest_flux_v1.safetensors`. Trigger: **`chtreal`**.

Dataset: `keepers/chest_real` (Drive folder `1Fti6dG9nydaIyhrzNXYOPDj_lke1iOjO`) plus real inbox `ADD_FLUX_CHEST` if present. **Never** `generate/` or `keepers/output` gens.

Train recipe: Ostris ai-toolkit like the male LoRA, but smaller (rank **8**, **800** steps, lr `1e-4`). Henry runs this cell himself. Keep the tab open.

**Run:** A100. Cells **1 → 2 → 3 → 4** then **Runtime → Run this cell only.**

### עברית
A100. תאים **1 → 2 → 3 → 4**. אחר כך **Runtime → Run this cell only** (רק תא 8). אימון LoRA חזה מתמונות אמיתיות בלבד. לא נוגע ב-v2. בלי תמונות מ-generate.


In [ ]:
# @title 8) Train small chest/nipple LoRA from chest_real only
# TRAIN cell. Real photos only. Does NOT overwrite lapetitemilf_flux_v2.
# Dataset: keepers/chest_real (+ ADD_FLUX_CHEST inbox). Never generate/ or keepers/output.
# Trigger: chtreal. Out: MyDrive/FiratSuper/loras/lapetitemilf_chest_flux_v1.safetensors
# Henry runs this cell himself. Keep the tab open.
import os
import gc
import glob
import shutil
import subprocess
import sys
import torch
from IPython.display import display, Image as IPyImage

CHEST_LORA_NAME = "lapetitemilf_chest_flux_v1"
CHEST_OUTPUT_LORA = CHEST_LORA_NAME + ".safetensors"
CHEST_TRIGGER = "chtreal"
CHEST_DATASET_DIR = "/content/dataset_chest"
CHEST_CONFIG_PATH = "/content/lapetitemilf_chest_flux.yaml"
CHEST_REAL_FOLDER_ID = "1Fti6dG9nydaIyhrzNXYOPDj_lke1iOjO"
# Cell 2 already defines FLUX_CHEST_ID / CHEST_DIR for ADD_FLUX_CHEST.
CHEST_RANK = 8
CHEST_ALPHA = 8
CHEST_TRAIN_STEPS = 800
CHEST_LR = 1e-4
CHEST_SAVE_EVERY = 200
DRY_ONLY = False  # True = 5-step dry run only, no full train / no Drive copy
DRY_STEPS = 5
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
SKIP_NAMES = {".drive_upload.json", ".ds_store", "thumbs.db"}
GEN_NAME_MARKS = (
    "_seed", "scene_", "strip_", "prepaste", "flux_eval", "hard_paste",
    "face_ok", "from17", "from18",
)
CAPTION = (
    "chtreal, close-up of an adult woman's bare chest, natural teardrop hang, "
    "medium circular pinkish-tan Montgomery-textured areolae, prominent nipples, "
    "fair pale skin, photorealistic photo"
)

if not SUBJECT_IS_ADULT:
    raise RuntimeError("Adult subject only.")
if CHEST_OUTPUT_LORA in PROTECTED_LORAS or CHEST_OUTPUT_LORA == OUTPUT_LORA_NAME:
    raise RuntimeError("Refusing to write a protected / v2 LoRA name.")
if CHEST_LORA_NAME == LORA_NAME:
    raise RuntimeError("Chest LoRA must not reuse the locked v2 name.")
if not os.path.isdir("/content/ai-toolkit"):
    raise RuntimeError("ai-toolkit missing. Run cells 1-4 first.")

# Free GPU if a generate pipe is still in RAM from cells 5-7.
for key in ("pipe", "img2img_pipe"):
    if key in globals() and globals()[key] is not None:
        try:
            del globals()[key]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def _banned_path(path):
    parts = path.replace("\\", "/").split("/")
    low = path.replace("\\", "/").lower()
    if "generate" in parts or any(p.startswith("generate") for p in parts):
        return "generate/"
    if "keepers" in parts and "output" in parts:
        return "keepers/output"
    if "flux_eval" in low or "/output/lapetitemilf/" in low:
        return "eval/output gens"
    return None


def _looks_generated(name):
    low = name.lower()
    return any(m in low for m in GEN_NAME_MARKS)


def _ensure_folder(local_dir, drive_name, folder_id):
    if os.path.isdir(local_dir) and any(
        os.path.splitext(n)[1].lower() in IMG_EXT for n in os.listdir(local_dir)
    ):
        return local_dir
    print("Local chest folder empty or missing:", local_dir)
    print("Downloading", drive_name, "id", folder_id)
    service = globals().get("DRIVE_SERVICE") or _api_service()
    found = None
    try:
        found = api_find_child(service, FIRATSUPER_DRIVE_ID, drive_name)
    except Exception as err:
        print("find", drive_name, "skipped:", err)
    use_id = found["id"] if found else folder_id
    api_download_folder(service, use_id, local_dir)
    return local_dir


chest_real_dir = os.path.join(KEEPERS_DIR, "chest_real")
_ensure_folder(chest_real_dir, "chest_real", CHEST_REAL_FOLDER_ID)
if os.path.isdir(os.path.join(KEEPERS_DIR, "chest_real")) is False:
    chest_real_dir = os.path.join(ROOT, "keepers", "chest_real")

# Canonical first: keepers/chest_real. Inbox ADD_FLUX_CHEST is extra real photos only.
source_folders = [
    ("keepers/chest_real", chest_real_dir),
    ("ADD_FLUX_CHEST", CHEST_DIR),
]
print("Canonical chest_real:", chest_real_dir)
print("Inbox ADD_FLUX_CHEST:", CHEST_DIR)
if not os.path.isdir(CHEST_DIR) or not os.listdir(CHEST_DIR):
    try:
        _ensure_folder(CHEST_DIR, "ADD_FLUX_CHEST", FLUX_CHEST_ID)
    except Exception as err:
        print("ADD_FLUX_CHEST download skipped:", err)

if os.path.isdir(CHEST_DATASET_DIR):
    shutil.rmtree(CHEST_DATASET_DIR)
os.makedirs(CHEST_DATASET_DIR, exist_ok=True)

pairs = []
skipped = []
seen = set()
for label, folder in source_folders:
    banned = _banned_path(os.path.abspath(folder) if folder else "")
    if banned:
        raise RuntimeError("Refusing dataset folder that looks like %s: %s" % (banned, folder))
    if not os.path.isdir(folder):
        print("WARNING: missing", label, folder)
        skipped.append(label + " (missing folder)")
        continue
    print("Scanning", label, folder)
    for name in sorted(os.listdir(folder)):
        if name.startswith(".") or name.lower() in SKIP_NAMES:
            continue
        path = os.path.join(folder, name)
        if os.path.isdir(path):
            skipped.append(label + "/" + name + "/ (subdir)")
            continue
        stem, ext = os.path.splitext(name)
        if ext.lower() not in IMG_EXT:
            continue
        if _looks_generated(name):
            skipped.append(label + "/" + name + " (looks generated, never train)")
            continue
        banned = _banned_path(path)
        if banned:
            skipped.append(label + "/" + name + " (banned " + banned + ")")
            continue
        if stem.lower() in seen:
            skipped.append(label + "/" + name + " (dup stem)")
            continue
        dest_img = os.path.join(CHEST_DATASET_DIR, name)
        dest_txt = os.path.join(CHEST_DATASET_DIR, stem + ".txt")
        shutil.copy2(path, dest_img)
        caption = CAPTION
        if not caption.lower().startswith(CHEST_TRIGGER):
            raise RuntimeError("Caption must start with " + CHEST_TRIGGER)
        if "ohwx woman" in caption.lower():
            raise RuntimeError("Chest captions must not retrain ohwx woman identity.")
        with open(dest_txt, "w", encoding="ascii") as fh:
            fh.write(caption + "\n")
        seen.add(stem.lower())
        pairs.append((label + "/" + name, caption, path))
        print("KEEP", label + "/" + name, os.path.getsize(path), "bytes")

print("Dataset count:", len(pairs))
if skipped:
    print("Skipped:")
    for row in skipped[:40]:
        print(" ", row)
if len(pairs) == 0:
    raise RuntimeError(
        "Cell 8 has 0 real chest photos. Check Drive keepers/chest_real "
        "(folder %s) and ADD_FLUX_CHEST. Will not train." % CHEST_REAL_FOLDER_ID
    )
if len(pairs) < 6:
    print(
        "WARNING: only %d real photos. Small-dataset risk (overfit / weak nipple lock). "
        "Henry should confirm chest_real has enough close-up real nipples." % len(pairs)
    )
print("--- sample caption ---")
print(pairs[0][0])
print(pairs[0][1])
print("----------------------")
print("Trigger:", CHEST_TRIGGER)
print("Local train folder:", CHEST_DATASET_DIR)
print("Will NOT write", OUTPUT_LORA_NAME)


def write_chest_yaml(path, steps, dry):
    sample_flag = "true" if dry else "false"
    save_every = 10000 if dry else CHEST_SAVE_EVERY
    lines = [
        "job: extension",
        "config:",
        '  name: "%s"' % CHEST_LORA_NAME,
        "  process:",
        "    - type: sd_trainer",
        '      training_folder: "%s"' % TRAIN_OUTPUT_DIR,
        "      device: cuda:0",
        '      trigger_word: "%s"' % CHEST_TRIGGER,
        "      network:",
        "        type: lora",
        "        linear: %d" % CHEST_RANK,
        "        linear_alpha: %d" % CHEST_ALPHA,
        "      save:",
        "        dtype: float16",
        "        save_every: %d" % save_every,
        "        max_step_saves_to_keep: 4",
        "        push_to_hub: false",
        "      datasets:",
        '        - folder_path: "%s"' % CHEST_DATASET_DIR,
        "          caption_ext: txt",
        "          caption_dropout_rate: 0.05",
        "          shuffle_tokens: false",
        "          cache_latents_to_disk: true",
        "          resolution: [512, 768, 1024]",
        "      train:",
        "        batch_size: 1",
        "        steps: %d" % steps,
        "        gradient_accumulation_steps: 1",
        "        train_unet: true",
        "        train_text_encoder: false",
        "        gradient_checkpointing: true",
        "        noise_scheduler: flowmatch",
        "        optimizer: adamw8bit",
        "        lr: %.0e" % CHEST_LR,
        "        skip_first_sample: true",
        "        disable_sampling: %s" % sample_flag,
        "        ema_config:",
        "          use_ema: true",
        "          ema_decay: 0.99",
        "        dtype: bf16",
        "      model:",
        '        name_or_path: "black-forest-labs/FLUX.1-dev"',
        "        is_flux: true",
        "        quantize: true",
        "      sample:",
        "        sampler: flowmatch",
        "        sample_every: 10000",
        "        sample_start_step: 0",
        "        width: 1024",
        "        height: 768",
        "        prompts:",
        '          - "chtreal, close-up bare chest, natural teardrop hang, medium circular pinkish-tan areolae, prominent nipples, photorealistic raw photo"',
        '        neg: ""',
        "        seed: 42",
        "        walk_seed: true",
        "        guidance_scale: 4",
        "        sample_steps: 20",
        "meta:",
        '  name: "[name]"',
        '  version: "1.0"',
        "",
    ]
    text = "\n".join(lines)
    if any(ord(ch) > 127 for ch in text):
        raise RuntimeError("YAML is not ASCII")
    if LORA_NAME in text or OUTPUT_LORA_NAME in text:
        raise RuntimeError("Chest YAML must not name the locked v2 LoRA.")
    if "generate/" in text or "keepers/output" in text:
        raise RuntimeError("Chest YAML must not point at generate/ or keepers/output.")
    with open(path, "w", encoding="ascii") as fh:
        fh.write(text)
    print("Wrote", path, "steps=%d dry=%s rank=%d" % (steps, dry, CHEST_RANK))


print("Chest train settings: rank", CHEST_RANK, "steps", CHEST_TRAIN_STEPS, "lr", CHEST_LR)
print("Mirror of male LoRA recipe, smaller (male was rank 16 / 2000 steps).")

n_txt = len([n for n in os.listdir(CHEST_DATASET_DIR) if n.endswith(".txt")])
if n_txt != len(pairs):
    raise RuntimeError("Caption count mismatch: images %d txt %d" % (len(pairs), n_txt))

write_chest_yaml(CHEST_CONFIG_PATH, DRY_STEPS, dry=True)
print("Dry run: %d steps (no samples)." % DRY_STEPS)
cmd = [sys.executable, "run.py", CHEST_CONFIG_PATH]
print("+", " ".join(cmd))
subprocess.check_call(cmd, cwd="/content/ai-toolkit")
print("Chest dry run OK.")

if DRY_ONLY:
    print("DRY_ONLY=True. Stopped after dry run. Set DRY_ONLY=False and rerun for full train.")
else:
    write_chest_yaml(CHEST_CONFIG_PATH, CHEST_TRAIN_STEPS, dry=False)
    print("Full chest train:", CHEST_TRAIN_STEPS, "steps on", torch.cuda.get_device_name(0))
    print("Writes /content/output/%s/  NOT  %s" % (CHEST_LORA_NAME, OUTPUT_LORA_NAME))
    print("Keep this tab open.")
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd="/content/ai-toolkit")
    print("Chest training finished.")

    run_dir = os.path.join(TRAIN_OUTPUT_DIR, CHEST_LORA_NAME)
    candidates = []
    final_path = os.path.join(run_dir, CHEST_OUTPUT_LORA)
    if os.path.isfile(final_path):
        candidates.append(final_path)
    candidates.extend(sorted(glob.glob(os.path.join(run_dir, "*.safetensors"))))
    preferred = [p for p in candidates if os.path.basename(p) == CHEST_OUTPUT_LORA]
    if not preferred:
        preferred = [p for p in candidates if "step" not in os.path.basename(p).lower()]
    if not preferred:
        preferred = candidates
    if not preferred:
        raise RuntimeError("No .safetensors found in " + run_dir)
    src = preferred[0]
    base = os.path.basename(src)
    if base in PROTECTED_LORAS or base == OUTPUT_LORA_NAME:
        raise RuntimeError("Refusing to copy a locked LoRA: " + base)
    dest_rel = "loras/" + CHEST_OUTPUT_LORA
    dest_abs = os.path.join(ROOT, dest_rel)
    if os.path.basename(dest_abs) in PROTECTED_LORAS:
        raise RuntimeError("Refusing to write protected LoRA dest.")
    print("Using:", src, "size_mb=%.1f" % (os.path.getsize(src) / 1024**2))
    upload_project_file(src, dest_rel)
    print("Chest LoRA saved as", dest_rel)
    print("Locked file was not touched:", OUTPUT_LORA_NAME)
    print("Trigger token:", CHEST_TRIGGER)
    print("Next: cell 9 generates with v2 @ 1.15 + this chest adapter.")
    print("Do not put generated pictures into chest_real or ADD_FLUX_CHEST.")

print("step2 Cell 8 done. Dataset", len(pairs), "real photos. Trigger", CHEST_TRIGGER)


# step2 Cell 9 — woman-only generate with v2 + chest LoRA

**step2 תא 9 — יצירה אישה בלבד עם v2 + LoRA החזה**

Run **after cell 8 finished** and `loras/lapetitemilf_chest_flux_v1.safetensors` exists. Woman-only. Face lock like cell 5 (`lapetitemilf_flux_v2` @ **1.15** + keeper names). Chest adapter `chtreal` at **0.60 / 0.75 / 0.90** (pick set). Topless. Photoreal. **No paste. No male LoRA.**

If you just trained, Runtime may need a restart: then **1 → 2 → 3 → 4** and this cell (pipe reload).

**Run:** A100. Cells **1 → 2 → 3 → 4** then **Runtime → Run this cell only.**

~12 stills. Seeds **8002, 8101, 8900, 8901** × three chest weights.
Writes `MyDrive/FiratSuper/generate/scene_step2_09_woman_chest_lora_<timestamp>/`.

### עברית
A100. אחרי שתא 8 סיים והקובץ קיים: תאים **1 → 2 → 3 → 4**, אחר כך **רק תא 9**. אישה בלבד, פנים מ-v2, חזה מ-`chtreal`. בלי הדבקה, בלי זוג.


In [ ]:
# @title 9) Woman-only generate with v2 + chest LoRA
# Generate-only. Do NOT train. Do NOT overwrite v2 or write new .safetensors.
# Face: locked lapetitemilf_flux_v2 @ 1.15 + cell 5 keeper names.
# Chest: lapetitemilf_chest_flux_v1 adapter "chest" @ 0.60 / 0.75 / 0.90.
# Trigger token from cell 8: chtreal. Topless. No paste. No male LoRA.
# Writes MyDrive/FiratSuper/generate/scene_step2_09_woman_chest_lora_<timestamp>/
import os
import torch
from datetime import datetime
from IPython.display import display

FACE_OK_SCENE66 = "1QZMUC79DFg3kPPLT51lES-Xfml0zTPIs"
SHOT_START = 0
SHOT_END = 12
SLUG = "step2_09_woman_chest_lora"
CHEST_LORA_NAME = "lapetitemilf_chest_flux_v1"
CHEST_OUTPUT_LORA = CHEST_LORA_NAME + ".safetensors"
CHEST_TRIGGER = "chtreal"
CHEST_ADAPTER = "chest"
LORA_W = 1.15
CHEST_WEIGHTS = [0.60, 0.75, 0.90]
SEEDS = [8002, 8101, 8900, 8901]
# 3 weights x 4 seeds = 12
JOBS = []
n = 1
for seed in SEEDS:
    for w in CHEST_WEIGHTS:
        JOBS.append((n, w, seed))
        n += 1
NEG = (
    "cartoon, illustration, anime, 3d render, cgi, painting, digital art, "
    "plastic skin, beauty filter, over-smoothed, airbrushed, "
    "bra, bikini top, lingerie covering chest, lace covering nipples, underwire, "
    "nipples through fabric, floating nipples on clothing, sheer bra, "
    "warped nipples, misshapen nipples, stringy nipples, "
    "glasses, gold necklace, black tank, gold curtains, "
    "chin crop, missing top of head, hrmale, man, couple, penis, "
    "multiple women, triplets, clone army, duplicate person"
)
IDENT = (
    "ohwx woman, adult woman, long highlighted blonde hair, brown eyes, "
    "head fully in frame, "
)
FACE = (
    "matching the face identity of keeper stills "
    "01_face_ok, 02_face_ok, 03_face_ok, 04_face_ok, "
    "and the accepted face in 02_stand_three_q_chests, "
    "17_scene81_seed8002_prepaste_ok, 18_scene81_seed8101_prepaste_ok, "
)
CHEST = (
    "chtreal, fair pale skin, slim torso, natural teardrop hang, "
    "medium circular pinkish-tan Montgomery-textured areolae, "
    "prominent nipples, matching chest_real, "
)
PLACE = (
    "full head in frame, space above the hair, waist-up medium shot, "
    "topless, bare breasts, no bra, no lingerie on chest, "
    "intimate bedroom, looking at the camera, "
    "photorealistic raw photo, natural skin texture, real camera photograph"
)
SHOTS = [
    "standing waist-up, topless, bare breasts, chest readable",
    "standing waist-up, topless, bare breasts, looking at the camera",
    "three-quarter waist-up, topless, bare chest, no bra",
    "standing waist-up, topless, bare breasts, indoor daylight",
]
BANNED = (
    "hrmale", "penis", "glans", "semen", "couple",
    "glasses", "gold necklace", "black tank", "gold curtains", "chin crop",
    "cartoon", "anime", "illustration",
    "underwire", "bikini top",
)

if len(JOBS) != 12:
    raise RuntimeError("step2 Cell 9 must be exactly 12 stills.")
if 8002 not in SEEDS or 8101 not in SEEDS:
    raise RuntimeError("step2 Cell 9 must include anchor seeds 8002 and 8101.")
if not SUBJECT_IS_ADULT:
    raise RuntimeError("Adult subject only.")
if abs(LORA_W - 1.15) > 1e-6:
    raise RuntimeError("step2 Cell 9 must keep v2 at 1.15.")
if any(w < 0.5 or w > 1.01 for w in CHEST_WEIGHTS):
    raise RuntimeError("Chest adapter weights should stay in a modest 0.6-0.9 band.")

v2_path = os.path.join(LORAS_DIR, OUTPUT_LORA_NAME)
if OUTPUT_LORA_NAME != "lapetitemilf_flux_v2.safetensors":
    raise RuntimeError("step2 Cell 9 must load locked lapetitemilf_flux_v2.")
if not os.path.isfile(v2_path):
    raise RuntimeError("Load-only: missing " + v2_path + " (will not train).")
chest_path = os.path.join(LORAS_DIR, CHEST_OUTPUT_LORA)
if not os.path.isfile(chest_path):
    local_final = os.path.join(TRAIN_OUTPUT_DIR, CHEST_LORA_NAME, CHEST_OUTPUT_LORA)
    if os.path.isfile(local_final):
        chest_path = local_final
if not os.path.isfile(chest_path):
    raise RuntimeError(
        "Chest LoRA missing: " + os.path.join(LORAS_DIR, CHEST_OUTPUT_LORA) +
        " Run cell 8 first and wait until it copies the file."
    )
if os.path.basename(chest_path) in PROTECTED_LORAS:
    raise RuntimeError("Chest path resolved to a protected LoRA. Stop.")
print("LOAD ONLY face", OUTPUT_LORA_NAME, "bytes", os.path.getsize(v2_path))
print("LOAD ONLY chest", chest_path, "bytes", os.path.getsize(chest_path))
v2_mtime, v2_size = os.path.getmtime(v2_path), os.path.getsize(v2_path)
print("v2 @ 1.15 + chest adapter", CHEST_ADAPTER, "weights", CHEST_WEIGHTS)
print("Trigger", CHEST_TRIGGER, "in the prompt. No paste. No male LoRA.")

ensure_flux_pipe()
if type(pipe).__name__ != "FluxPipeline":
    raise RuntimeError("Need Flux txt2img pipe. Restart runtime, rerun 1-4, then this cell.")
used_v2 = set_pipe_adapters(pipe, ["default"], [LORA_W])
if not getattr(pipe, "_chest_adapter_loaded", False):
    print("Loading chest LoRA as adapter", CHEST_ADAPTER, chest_path)
    try:
        pipe.load_lora_weights(chest_path, adapter_name=CHEST_ADAPTER)
    except TypeError as err:
        raise RuntimeError(
            "This diffusers build cannot stack two LoRAs (%s). "
            "Runtime > Restart session, rerun cells 1-4, then 9." % err
        )
    pipe._chest_adapter_loaded = True
print("Female-only. Face adapter", used_v2, "@", LORA_W, "+ chest", CHEST_ADAPTER)
print("Male LoRA not loaded for this cell.")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
gen_parent = os.path.join(ROOT, "generate")
parts = gen_parent.split(os.sep)
if "keepers" in parts or "loras" in parts or any(p.startswith("ADD_") for p in parts):
    raise RuntimeError("Refusing to write under keepers/loras/ADD_*")
os.makedirs(gen_parent, exist_ok=True)
out_dir = os.path.join(gen_parent, "scene_" + SLUG + "_" + stamp)
os.makedirs(out_dir, exist_ok=True)
print("Out dir:", out_dir)
print("Drive path: MyDrive/FiratSuper/generate/" + os.path.basename(out_dir))

saved = []
for n, chest_w, seed in JOBS:
    action = SHOTS[(n - 1) % len(SHOTS)]
    used = set_pipe_adapters(pipe, ["default", CHEST_ADAPTER], [LORA_W, chest_w])
    prompt = IDENT + FACE + CHEST + action + ". " + PLACE
    low = prompt.lower()
    hit = [w for w in BANNED if w in low]
    if hit:
        raise RuntimeError("step2 Cell 9 prompt has banned words: " + ", ".join(hit))
    words = [w.strip(".,;:") for w in low.split()]
    if "man" in words or "couple" in words or "hrmale" in words:
        raise RuntimeError("step2 Cell 9 must stay woman-only.")
    for lock in (
        "ohwx woman",
        "chtreal",
        "01_face_ok",
        "04_face_ok",
        "17_scene81_seed8002_prepaste_ok",
        "18_scene81_seed8101_prepaste_ok",
        "fair pale skin",
        "natural teardrop hang",
        "medium circular pinkish-tan Montgomery-textured areolae",
        "prominent nipples",
        "chest_real",
        "topless",
        "bare breasts",
        "photorealistic raw photo",
    ):
        if lock not in prompt and lock not in low:
            raise RuntimeError("step2 Cell 9 prompt missing lock: " + lock)
    clip_tok = getattr(pipe, "tokenizer", None)
    if clip_tok is not None:
        clip_n = len(clip_tok(prompt, add_special_tokens=True).input_ids)
        print("CLIP tokens", clip_n, "(info only; do not shorten)")
    wtag = "w%03d" % int(round(chest_w * 100))
    fname = "%02d_%s_seed%d.png" % (n, wtag, seed)
    print("---", fname, "v2", LORA_W, "chest", chest_w, "seed", seed, "adapters", used)
    print(prompt)
    image = pipe(
        prompt=prompt,
        negative_prompt=NEG,
        guidance_scale=3.5,
        height=768,
        width=1024,
        num_inference_steps=32,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]
    path = os.path.join(out_dir, fname)
    image.save(path)
    saved.append(path)
    print("saved", path, "chest_weight", chest_w)
    display(image)

if os.path.getmtime(v2_path) != v2_mtime or os.path.getsize(v2_path) != v2_size:
    raise RuntimeError("v2 LoRA file changed during generate. Stop.")
print("Saved", len(saved), "stills in", out_dir)
print("Drive path: MyDrive/FiratSuper/generate/" + os.path.basename(out_dir))
if USE_DRIVE_API:
    for path in saved:
        if path.endswith(".safetensors"):
            raise RuntimeError("Refusing to upload safetensors from step2 Cell 9.")
        upload_project_file(path, os.path.relpath(path, ROOT))
fid = None
try:
    service = DRIVE_SERVICE or _api_service()
    gen_folder = api_ensure_folder(service, FIRATSUPER_DRIVE_ID, "generate")
    found = api_find_child(service, gen_folder, os.path.basename(out_dir))
    fid = found["id"] if found else gen_folder
    print("Drive folder id:", fid)
except Exception as err:
    print("Drive folder id lookup skipped:", err)
print("SCENE_STEP2_09_DIR", out_dir)
print("step2 Cell 9 done. v2 unchanged. Chest adapter weights", CHEST_WEIGHTS)
print("Do not put these pictures back into chest_real, ADD_FLUX_CHEST, or ADD_*.")
